In [ ]:
#FOR ITERATION VERSION 1 - RAW
def _get_ext_grid_results(net, ppci_0, ppci_1, ppci_2, bus):
    bus_lookup = net._pd2ppc_lookups["bus"]
    bus_fault_ppc = bus_lookup[bus]
    ext_contrib = []

    for i, eg in net.ext_grid.iterrows():
        bus_ext_ppc = bus_lookup[eg.bus]
        vn_net = net.bus.loc[eg.bus, "vn_kv"]
        fault = net["_options"]["fault"]

        # --- IEC 60909 - impedance of external network ---
        s_sc = eg.s_sc_max_mva
        rx = eg.rx_max
        c = 1.1

        ZQ = (vn_net ** 2) / s_sc
        RQ = ZQ * rx / np.sqrt(1 + rx ** 2)
        XQ = ZQ / np.sqrt(1 + rx ** 2)
        ZQ_complex = complex(RQ, XQ)*c

        # --- Equivalent voltage source (Thevenin source) ---
        Eq = c * vn_net / np.sqrt(3)

        # === Fault-specific calculations ===
        if fault == "LLL":
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z1 = ZQ_complex
            else:
                # Find branch between ext_grid bus and fault bus
                mask = ((ppci_1['branch'][:, 0] == bus_fault_ppc) & (ppci_1['branch'][:, 1] == bus_ext_ppc)) | \
                       ((ppci_1['branch'][:, 1] == bus_fault_ppc) & (ppci_1['branch'][:, 0] == bus_ext_ppc))
                if np.any(mask):
                    br = ppci_1['branch'][mask][0]
                    Zbr1 = br[BR_R] + 1j * br[BR_X]
                    Z1 = Zbr1 * baseZ1 + ZQ_complex
                else:
                    print(f"[WARN] No direct branch found between buses {bus_fault_ppc} and {bus_ext_ppc}")
                    Z1 = ZQ_complex

            Z1 = complex(np.asarray(Z1).squeeze())
            I1 = Eq / Z1 if Z1 != 0 and not np.isnan(Z1) else complex(np.nan)
            I2 = I0 = 0j

        elif fault == "LG":
            baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
            baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z0, Z1, Z2 = ZQ_complex, ZQ_complex, ZQ_complex
            else:
                def Zbr(ppci, baseZ):
                    mask = ((ppci['branch'][:, 0] == bus_fault_ppc) & (ppci['branch'][:, 1] == bus_ext_ppc)) | \
                           ((ppci['branch'][:, 1] == bus_fault_ppc) & (ppci['branch'][:, 0] == bus_ext_ppc))
                    if np.any(mask):
                        br = ppci['branch'][mask][0]
                        return (br[BR_R] + 1j * br[BR_X]) * baseZ
                    else:
                        print(f"[WARN] No direct branch between buses {bus_fault_ppc} and {bus_ext_ppc}")
                        return 0j

                Z1 = Zbr(ppci_1, baseZ1) + ZQ_complex
                Z2 = Zbr(ppci_2, baseZ2) + ZQ_complex
                Z0 = Zbr(ppci_0, baseZ0) + ZQ_complex

            Zeq = Z0 + Z1 + Z2
            I0 = I1 = I2 = Eq / Zeq if Zeq != 0 and not np.isnan(Zeq) else complex(np.nan)

        elif fault == "LL":
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
            baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z1 = Z2 = ZQ_complex
            else:
                def Zbr(ppci, baseZ):
                    mask = ((ppci['branch'][:, 0] == bus_fault_ppc) & (ppci['branch'][:, 1] == bus_ext_ppc)) | \
                           ((ppci['branch'][:, 1] == bus_fault_ppc) & (ppci['branch'][:, 0] == bus_ext_ppc))
                    if np.any(mask):
                        br = ppci['branch'][mask][0]
                        return (br[BR_R] + 1j * br[BR_X]) * baseZ
                    else:
                        print(f"[WARN] No direct branch between buses {bus_fault_ppc} and {bus_ext_ppc}")
                        return 0j

                Z1 = Zbr(ppci_1, baseZ1) + ZQ_complex
                Z2 = Zbr(ppci_2, baseZ2) + ZQ_complex

            Zeq = Z1 + Z2
            I1 = Eq / Zeq if Zeq != 0 and not np.isnan(Zeq) else complex(np.nan)
            I2 = -I1
            I0 = 0j

        elif fault == "LLG":
            baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
            baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z0 = Z1 = Z2 = ZQ_complex
            else:
                def Zbr(ppci, baseZ):
                    mask = ((ppci['branch'][:, 0] == bus_fault_ppc) & (ppci['branch'][:, 1] == bus_ext_ppc)) | \
                           ((ppci['branch'][:, 1] == bus_fault_ppc) & (ppci['branch'][:, 0] == bus_ext_ppc))
                    if np.any(mask):
                        br = ppci['branch'][mask][0]
                        return (br[BR_R] + 1j * br[BR_X]) * baseZ
                    else:
                        print(f"[WARN] No direct branch between buses {bus_fault_ppc} and {bus_ext_ppc}")
                        return 0j

                Z1 = Zbr(ppci_1, baseZ1) + ZQ_complex
                Z2 = Zbr(ppci_2, baseZ2) + ZQ_complex
                Z0 = Zbr(ppci_0, baseZ0) + ZQ_complex

            Zeq_part = (Z2 * Z0) / (Z2 + Z0)
            Zeq = Z1 + Zeq_part
            I1 = Eq / Zeq if Zeq != 0 and not np.isnan(Zeq) else complex(np.nan)
            I2 = -I1 * Z0 / (Z2 + Z0)
            I0 = -I1 * Z2 / (Z2 + Z0)

        else:
            raise ValueError(f"Unsupported fault type: {fault}")

        # --- Phase conversion ---
        for name in ["I0", "I1", "I2"]:
            val = locals()[name]
            try:
                locals()[name] = complex(np.asarray(val).squeeze())
            except Exception:
                arr = np.asarray(val)
                if arr.size == 1:
                    locals()[name] = complex(arr.flatten()[0])
                else:
                    raise ValueError(f"{name} is non-scalar: shape={arr.shape}, value={val}")

        def to_scalar_complex(val, name):
            # Pretvori sve u python complex
            if np.isscalar(val):
                return complex(val)
            arr = np.asarray(val).squeeze()
            if arr.size == 1:
                return complex(arr.item())
            elif arr.ndim == 0:
                return complex(arr)
            else:
                print(f"[WARN] {name} non-scalar shape={arr.shape}, taking first element.")
                return complex(np.ravel(arr)[0])

        I0 = to_scalar_complex(I0, "I0")
        I1 = to_scalar_complex(I1, "I1")
        I2 = to_scalar_complex(I2, "I2")
        I_phase = sequence_to_phase(np.array([I0, I1, I2], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase[0], I_phase[1], I_phase[2]
        skss = vn_net / np.sqrt(3) * (abs(I_a) + abs(I_b) + abs(I_c))

        ext_contrib.append({
            "bus": eg.bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "skss_mva": skss,
        })

    net.res_ext_grid_sc = pd.DataFrame(ext_contrib)

In [ ]:
#VERSION 1.1 - FOR ITERATION + PPCI STRUCTURE MORE PRESENT
def _get_ext_grid_results(net, ppci_0, ppci_1, ppci_2, bus):
    bus_lookup = net._pd2ppc_lookups["bus"]
    bus_fault_ppc = bus_lookup[bus]
    ext_contrib = []

    for i, eg in net.ext_grid.iterrows():
        bus_ext_ppc = bus_lookup[eg.bus]
        vn_net = net.bus.loc[eg.bus, "vn_kv"]
        fault = net["_options"]["fault"]

        # --- IEC 60909 - impedance of external network ---
        s_sc = eg.s_sc_max_mva
        rx = eg.rx_max
        c = ppci_1["bus"][bus, C_MAX]
        kg= ppci_1["bus"][bus, K_G]

        ZQ = (vn_net ** 2) / s_sc
        RQ = ZQ * rx / np.sqrt(1 + rx ** 2)
        XQ = ZQ / np.sqrt(1 + rx ** 2)
        ZQ_complex = complex(RQ, XQ)
        z_exgk1 = ZQ_complex * kg

        # --- Equivalent voltage source (Thevenin source) ---
        Eq = c * vn_net / np.sqrt(3)

        # === Fault-specific calculations ===
        if fault == "LLL":
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z1 = z_exgk1
            else:
                # Find branch between ext_grid bus and fault bus
                mask = ((ppci_1['branch'][:, 0] == bus_fault_ppc) & (ppci_1['branch'][:, 1] == bus_ext_ppc)) | \
                       ((ppci_1['branch'][:, 1] == bus_fault_ppc) & (ppci_1['branch'][:, 0] == bus_ext_ppc))
                if np.any(mask):
                    br = ppci_1['branch'][mask][0]
                    Zbr1 = br[BR_R] + 1j * br[BR_X]
                    Z1 = Zbr1 * baseZ1 + z_exgk1
                else:
                    print(f"[WARN] No direct branch found between buses {bus_fault_ppc} and {bus_ext_ppc}")
                    Z1 = ZQ_complex

            Z1 = complex(np.asarray(Z1).squeeze())
            I1 = Eq / Z1 if Z1 != 0 and not np.isnan(Z1) else complex(np.nan)
            I2 = I0 = 0j

        elif fault == "LG":
            baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
            baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z0, Z1, Z2 = z_exgk1, z_exgk1, z_exgk1
            else:
                def Zbr(ppci, baseZ):
                    mask = ((ppci['branch'][:, 0] == bus_fault_ppc) & (ppci['branch'][:, 1] == bus_ext_ppc)) | \
                           ((ppci['branch'][:, 1] == bus_fault_ppc) & (ppci['branch'][:, 0] == bus_ext_ppc))
                    if np.any(mask):
                        br = ppci['branch'][mask][0]
                        return (br[BR_R] + 1j * br[BR_X]) * baseZ
                    else:
                        print(f"[WARN] No direct branch between buses {bus_fault_ppc} and {bus_ext_ppc}")
                        return 0j

                Z1 = Zbr(ppci_1, baseZ1) + z_exgk1
                Z2 = Zbr(ppci_2, baseZ2) + z_exgk1
                Z0 = Zbr(ppci_0, baseZ0) + z_exgk1

            Zeq = Z0 + Z1 + Z2
            I0 = I1 = I2 = Eq / Zeq if Zeq != 0 and not np.isnan(Zeq) else complex(np.nan)

        elif fault == "LL":
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
            baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z1 = Z2 = z_exgk1
            else:
                def Zbr(ppci, baseZ):
                    mask = ((ppci['branch'][:, 0] == bus_fault_ppc) & (ppci['branch'][:, 1] == bus_ext_ppc)) | \
                           ((ppci['branch'][:, 1] == bus_fault_ppc) & (ppci['branch'][:, 0] == bus_ext_ppc))
                    if np.any(mask):
                        br = ppci['branch'][mask][0]
                        return (br[BR_R] + 1j * br[BR_X]) * baseZ
                    else:
                        print(f"[WARN] No direct branch between buses {bus_fault_ppc} and {bus_ext_ppc}")
                        return 0j

                Z1 = Zbr(ppci_1, baseZ1) + z_exgk1
                Z2 = Zbr(ppci_2, baseZ2) + z_exgk1

            Zeq = Z1 + Z2
            I1 = Eq / Zeq if Zeq != 0 and not np.isnan(Zeq) else complex(np.nan)
            I2 = -I1
            I0 = 0j

        elif fault == "LLG":
            baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]
            baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
            baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

            if bus_ext_ppc == bus_fault_ppc:
                Z0 = Z1 = Z2 = z_exgk1
            else:
                def Zbr(ppci, baseZ):
                    mask = ((ppci['branch'][:, 0] == bus_fault_ppc) & (ppci['branch'][:, 1] == bus_ext_ppc)) | \
                           ((ppci['branch'][:, 1] == bus_fault_ppc) & (ppci['branch'][:, 0] == bus_ext_ppc))
                    if np.any(mask):
                        br = ppci['branch'][mask][0]
                        return (br[BR_R] + 1j * br[BR_X]) * baseZ
                    else:
                        print(f"[WARN] No direct branch between buses {bus_fault_ppc} and {bus_ext_ppc}")
                        return 0j

                Z1 = Zbr(ppci_1, baseZ1) + z_exgk1
                Z2 = Zbr(ppci_2, baseZ2) + z_exgk1
                Z0 = Zbr(ppci_0, baseZ0) + z_exgk1

            Zeq_part = (Z2 * Z0) / (Z2 + Z0)
            Zeq = Z1 + Zeq_part
            I1 = Eq / Zeq if Zeq != 0 and not np.isnan(Zeq) else complex(np.nan)
            I2 = -I1 * Z0 / (Z2 + Z0)
            I0 = -I1 * Z2 / (Z2 + Z0)

        else:
            raise ValueError(f"Unsupported fault type: {fault}")

        # --- Phase conversion ---
        for name in ["I0", "I1", "I2"]:
            val = locals()[name]
            try:
                locals()[name] = complex(np.asarray(val).squeeze())
            except Exception:
                arr = np.asarray(val)
                if arr.size == 1:
                    locals()[name] = complex(arr.flatten()[0])
                else:
                    raise ValueError(f"{name} is non-scalar: shape={arr.shape}, value={val}")

        def to_scalar_complex(val, name):
            # Pretvori sve u python complex
            if np.isscalar(val):
                return complex(val)
            arr = np.asarray(val).squeeze()
            if arr.size == 1:
                return complex(arr.item())
            elif arr.ndim == 0:
                return complex(arr)
            else:
                print(f"[WARN] {name} non-scalar shape={arr.shape}, taking first element.")
                return complex(np.ravel(arr)[0])

        I0 = to_scalar_complex(I0, "I0")
        I1 = to_scalar_complex(I1, "I1")
        I2 = to_scalar_complex(I2, "I2")
        I_phase = sequence_to_phase(np.array([I0, I1, I2], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase[0], I_phase[1], I_phase[2]
        V_phase_kv = vn_net / np.sqrt(3)
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva
        ext_contrib.append({
            "bus": eg.bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss": skss, "skss_a_mva": skss_a_mva,
            "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c))
        })

    net.res_ext_grid_sc = pd.DataFrame(ext_contrib)

In [ ]:
#VERSION 1.2 - FOR ITERATION + IMPEDANCE  APPROACH FROM GENERATOR STAND POINT - works for every case
def _get_ext_grid_results(net, ppci_0, ppci_1, ppci_2, bus):
    bus_lookup = net._pd2ppc_lookups["bus"]
    bus_fault_ppc = bus_lookup[bus]
    ext_contrib = []

    for i, eg in net.ext_grid.iterrows():

        if not eg.get("in_service", True):
            ext_contrib.append({
                "bus": eg.bus,
                "ikss_a_ka": np.nan, "ikss_b_ka": np.nan, "ikss_c_ka": np.nan,
                "ikss_a_degree": np.nan, "ikss_b_degree": np.nan, "ikss_c_degree": np.nan,
                "skss": np.nan, "skss_a_mva": np.nan, "skss_b_mva": np.nan, "skss_c_mva": np.nan,
                "p_a_mw": np.nan, "q_a_mvar": np.nan,
                "p_b_mw": np.nan, "q_b_mvar": np.nan,
                "p_c_mw": np.nan, "q_c_mvar": np.nan
            })
            continue  # preskoči dalji izračun za ovu eksternu mrežu

        bus_ext_ppc = bus_lookup[eg.bus]
        #bus_ext_ppc = bus_lookup[net.gen.bus.values]
        vn_net = net.bus.loc[eg.bus, "vn_kv"]
        #vn_net = net.bus.loc[bus_ext_ppc, "vn_kv"].values
        fault = net["_options"]["fault"]

        same_bus = (bus_ext_ppc  == bus_fault_ppc)
        # --- IEC 60909 - impedance of external network ---
        s_sc = eg.s_sc_max_mva
        rx = eg.rx_max
        c = ppci_1["bus"][bus, C_MAX]
        kg= ppci_1["bus"][bus, K_G]


        #kg = np.real(seq_params[:, 1, 2])
        Eg = 1.1 * vn_net/ np.sqrt(3)


        ZQ = (vn_net ** 2) / s_sc
        RQ = ZQ * rx / np.sqrt(1 + rx ** 2)
        XQ = ZQ / np.sqrt(1 + rx ** 2)
        ZQ_complex = complex(RQ, XQ)
        z_exgk1 = ZQ_complex * 1.1




        baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
        baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]
        baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]

        def Zbr(ppci, baseZ, bus_fault_ppc, bus_ext_ppc):
           Zbus = ppci['internal']['Zbus']
           if ppci=='ppci_0':
               Zeq = Zbus[bus_fault_ppc, bus_fault_ppc]
           else:
               Zeq = Zbus[bus_fault_ppc, bus_fault_ppc] + Zbus[bus_ext_ppc, bus_ext_ppc] - 2 * Zbus[bus_fault_ppc, bus_ext_ppc]
                # Convert back to Ohms
           return Zeq * baseZ

        Zb0 = Zbr(ppci_0, baseZ0, bus_fault_ppc, bus_ext_ppc)
        Zb1 = Zbr(ppci_1, baseZ1, bus_fault_ppc, bus_ext_ppc)
        Zb2 = Zbr(ppci_2, baseZ2, bus_fault_ppc, bus_ext_ppc)

        Z1 = np.where(bus_ext_ppc  == bus_fault_ppc, z_exgk1, Zb1 + z_exgk1)
        Z2 = np.where(bus_ext_ppc  == bus_fault_ppc, z_exgk1, Zb2 + z_exgk1)
        Z0 = np.where(bus_ext_ppc  == bus_fault_ppc, z_exgk1, Zb0 + z_exgk1)

        fault_type = net["_options"]["fault"]
        # === Fault-specific calculations ===
        if fault_type == "LLL":
            I1 = Eg / Z1
            I0 = I2 = np.zeros_like(I1)

        elif fault_type == "LG":
            I0 = I1 = I2 = Eg / (Z1 + Z2 + Z0)

        elif fault_type == "LL":
            I1 = Eg / (Z1 + Z2)
            I2 = -I1
            I0 = np.zeros_like(I1)

        elif fault_type == "LLG":
            I1 = Eg / (Z1 + (Z2 * Z0) / (Z2 + Z0))
            I2 = -I1 * Z0 / (Z2 + Z0)
            I0 = -I1 * Z2 / (Z2 + Z0)

        # --- Phase conversion ---
        for name in ["I0", "I1", "I2"]:
            val = locals()[name]
            try:
                locals()[name] = complex(np.asarray(val).squeeze())
            except Exception:
                arr = np.asarray(val)
                if arr.size == 1:
                    locals()[name] = complex(arr.flatten()[0])
                else:
                    raise ValueError(f"{name} is non-scalar: shape={arr.shape}, value={val}")

        def to_scalar_complex(val, name):
            # Pretvori sve u python complex
            if np.isscalar(val):
                return complex(val)
            arr = np.asarray(val).squeeze()
            if arr.size == 1:
                return complex(arr.item())
            elif arr.ndim == 0:
                return complex(arr)
            else:
                print(f"[WARN] {name} non-scalar shape={arr.shape}, taking first element.")
                return complex(np.ravel(arr)[0])

        I0 = to_scalar_complex(I0, "I0")
        I1 = to_scalar_complex(I1, "I1")
        I2 = to_scalar_complex(I2, "I2")
        I_phase = sequence_to_phase(np.array([I0, I1, I2], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase[0], I_phase[1], I_phase[2]
        V_phase_kv = vn_net / np.sqrt(3)
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva
        ext_contrib.append({
            "bus": eg.bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss": skss, "skss_a_mva": skss_a_mva,
            "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c))
        })

    net.res_ext_grid_sc = pd.DataFrame(ext_contrib)

In [ ]:
#VECTORIZED VERSION 2.0 - WORKS FOR EVERY CASE SCENARIO
def _get_ext_grid_results(net, ppci_0, ppci_1, ppci_2, bus):
    bus_lookup = net._pd2ppc_lookups["bus"]
    bus_fault_ppc = bus_lookup[bus]

    # Filter in-service external grids
    ext_grids = net.ext_grid[net.ext_grid.get("in_service", True)].copy()
    if ext_grids.empty:
        net.res_ext_grid_sc = pd.DataFrame([{
            "bus": np.nan,
            "ikss_a_ka": np.nan, "ikss_b_ka": np.nan, "ikss_c_ka": np.nan,
            "ikss_a_degree": np.nan, "ikss_b_degree": np.nan, "ikss_c_degree": np.nan,
            "skss": np.nan, "skss_a_mva": np.nan, "skss_b_mva": np.nan, "skss_c_mva": np.nan,
            "p_a_mw": np.nan, "q_a_mvar": np.nan,
            "p_b_mw": np.nan, "q_b_mvar": np.nan,
            "p_c_mw": np.nan, "q_c_mvar": np.nan
        }])
        return

    #Vector preparation
    bus_ext_ppc = bus_lookup[ext_grids.bus.values]
    vn_net = net.bus.loc[ext_grids.bus.values, "vn_kv"].values
    s_sc = ext_grids.s_sc_max_mva.values
    rx = ext_grids.rx_max.values
    fault_type = net["_options"]["fault"]
    cmax=ppci_1["bus"][bus_ext_ppc, C_MAX]

    # --- Constants per external grid
    Eg = cmax * vn_net / sqrt(3)
    ZQ = (vn_net ** 2) / s_sc
    RQ = ZQ * rx / np.sqrt(1 + rx ** 2)
    XQ = ZQ / np.sqrt(1 + rx ** 2)
    ZQ_complex = RQ + 1j * XQ
    t= 5.5
    z_exgk1 = z_exgk2 = z_exgk0 = ZQ_complex * cmax

    # Base impedances
    baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
    baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]
    baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]

    # Vectorized Zbus extraction
    def Zbr(ppci, baseZ, bus_fault_ppc, bus_ext_ppc):
        Zbus = ppci['internal']['Zbus']
        Z_from_from = Zbus[bus_fault_ppc, bus_fault_ppc]
        Z_to_to = np.diag(Zbus[bus_ext_ppc][:, bus_ext_ppc])
        Z_from_to = Zbus[bus_fault_ppc, bus_ext_ppc]
        Zeq = Z_from_from + Z_to_to - 2 * Z_from_to
        Zeq[np.isclose(Zeq, 0)] = 1e-12
        return Zeq * baseZ

    Zb0 = Zbr(ppci_0, baseZ0, bus_fault_ppc, bus_ext_ppc)
    Zb1 = Zbr(ppci_1, baseZ1, bus_fault_ppc, bus_ext_ppc)
    Zb2 = Zbr(ppci_2, baseZ2, bus_fault_ppc, bus_ext_ppc)


    same_bus = (bus_ext_ppc == bus_fault_ppc)
    Z1 = np.where(same_bus, z_exgk1, Zb1 + z_exgk1)
    Z2 = np.where(same_bus, z_exgk2, Zb2 + z_exgk2)
    Z0 = np.where(same_bus, z_exgk0, Zb0 + z_exgk0)

    # --- Fault current calculations (vectorized)
    if fault_type == "LLL":
        I1 = Eg / Z1
        I0 = I2 = np.zeros_like(I1)
    elif fault_type == "LG":
        I0 = I1 = I2 = Eg / (Z1 + Z2 + Z0)
    elif fault_type == "LL":
        I1 = Eg / (Z1 + Z2)
        I2 = -I1
        I0 = np.zeros_like(I1)
    elif fault_type == "LLG":
        I1 = Eg / (Z1 + (Z2 * Z0) / (Z2 + Z0))
        I2 = -I1 * Z0 / (Z2 + Z0)
        I0 = -I1 * Z2 / (Z2 + Z0)

    # --- Convert to phase currents and calculate results
    results = []
    for i in range(len(bus_ext_ppc)):
        I_phase = sequence_to_phase(np.array([I0[i], I1[i], I2[i]], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase
        V_phase_kv = vn_net[i] / np.sqrt(3)
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva

        results.append({
            "bus": ext_grids.bus.values[i],
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss": skss,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
        })

    net.res_ext_grid_sc = pd.DataFrame(results)


In [ ]:
#VECTORIZED VERSION 2.2 - transformer interaction - zERO IMPEDANCE IS IS STILL NOT DONE
def _get_ext_grid_results(net, ppci_0, ppci_1, ppci_2, bus):

    # --- Map the PandaPower bus index to the corresponding PPC bus index
    bus_lookup = net._pd2ppc_lookups["bus"]
    bus_fault_ppc = bus_lookup[bus]

    # --- Select all in-service external grids
    ext_grids = net.ext_grid[net.ext_grid.get("in_service", True)].copy()
    if ext_grids.empty:
        # No external grids found – return NaN placeholders for results
        net.res_ext_grid_sc = pd.DataFrame([{
            "bus": np.nan,
            "ikss_a_ka": np.nan, "ikss_b_ka": np.nan, "ikss_c_ka": np.nan,
            "ikss_a_degree": np.nan, "ikss_b_degree": np.nan, "ikss_c_degree": np.nan,
            "skss": np.nan, "skss_a_mva": np.nan, "skss_b_mva": np.nan, "skss_c_mva": np.nan,
            "p_a_mw": np.nan, "q_a_mvar": np.nan,
            "p_b_mw": np.nan, "q_b_mvar": np.nan,
            "p_c_mw": np.nan, "q_c_mvar": np.nan
        }])
        return

    # --- Gather external grid parameters
    bus_ext_ppc = bus_lookup[ext_grids.bus.values]
    vn_net = net.bus.loc[ext_grids.bus.values, "vn_kv"].values
    s_sc = ext_grids.s_sc_max_mva.values
    rx = ext_grids.rx_max.values
    fault_type = net["_options"]["fault"]
    cmax = ppci_1["bus"][bus_ext_ppc, C_MAX]

    # --- Calculate equivalent source voltage (Eg) and impedance (ZQ)
    Eg = cmax * vn_net / np.sqrt(3)
    ZQ = (vn_net ** 2) / s_sc
    RQ = ZQ * rx / np.sqrt(1 + rx ** 2)
    XQ = ZQ / np.sqrt(1 + rx ** 2)
    ZQ_complex = cmax * (RQ + 1j * XQ)

    # --- Calculate base impedances for each sequence network
    baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]
    baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
    baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

    # --- Helper: Compute equivalent impedance between fault bus and external grid bus
    def Zbr(ppci, baseZ, bus_fault_ppc, bus_ext_ppc):
        Zbus = ppci["internal"]["Zbus"]
        Z_from_from = Zbus[bus_fault_ppc, bus_fault_ppc]

        # Handle single or multiple external grid buses
        if np.isscalar(bus_ext_ppc):
            Z_to_to = Zbus[bus_ext_ppc, bus_ext_ppc]
            Z_from_to = Zbus[bus_fault_ppc, bus_ext_ppc]
        else:
            Z_to_to = np.diag(Zbus[bus_ext_ppc][:, bus_ext_ppc])
            Z_from_to = Zbus[bus_fault_ppc, bus_ext_ppc]

        Zeq = Z_from_from + Z_to_to - 2 * Z_from_to
        Zeq = np.atleast_1d(Zeq)
        Zeq[np.isclose(Zeq, 0)] = 1e-12  # Avoid numerical singularities
        return Zeq * baseZ

    # --- Check for transformers connected to the external grid
    trafo = net.trafo[
        net.trafo["hv_bus"].isin(ext_grids.bus.values) |
        net.trafo["lv_bus"].isin(ext_grids.bus.values)
    ]

    if not trafo.empty:
        # --- Extract transformer data (assuming one transformer per ext. grid)
        tr = trafo.iloc[0]
        vn_hv, vn_lv = float(tr.vn_hv_kv), float(tr.vn_lv_kv)
        sn, vk, vkr = float(tr.sn_mva), float(tr.vk_percent), float(tr.vkr_percent)

        # Transformer nominal ratio
        t_nominal = vn_hv / vn_lv
        branch = ppci_1["branch"]
        hv_bus, lv_bus = bus_lookup[tr.hv_bus], bus_lookup[tr.lv_bus]

        # Find transformer branch (for tap correction)
        mask = ((branch[:, F_BUS] == hv_bus) & (branch[:, T_BUS] == lv_bus)) | \
               ((branch[:, F_BUS] == lv_bus) & (branch[:, T_BUS] == hv_bus))
        KT_value = branch[mask, K_T] if np.any(mask) else 1.0

        # --- Transformer impedance (referenced to LV side)
        Zbase_lv = (vn_lv ** 2) / sn
        ZT = (vk / 100) * Zbase_lv
        RT = (vkr / 100) * Zbase_lv
        XT = np.sqrt(ZT ** 2 - RT ** 2)
        Z_T = RT + 1j * XT
        ZTLVK = KT_value * Z_T
        ZQt = ZQ_complex / (t_nominal ** 2)  # Refer network impedance to LV side

        # --- Total equivalent impedance (network + transformer)
        z_exgk1 = z_exgk2 = z_exgk0 = ZQt + ZTLVK
        #todo - z_exgk0 is special case that needs to be done according to IEC TR 60909-4

        # Determine if fault is on LV side of transformer
        if lv_bus <= bus_fault_ppc:
            Eg = Eg / t_nominal  # Adjust voltage to LV side
            Zb0, Zb1, Zb2 = (Zbr(ppci_0, baseZ0, bus_fault_ppc, lv_bus),
                             Zbr(ppci_1, baseZ1, bus_fault_ppc, lv_bus),
                             Zbr(ppci_2, baseZ2, bus_fault_ppc, lv_bus))
            same_bus = (bus_ext_ppc + 1 == bus_fault_ppc)
        else:
            # Fault before transformer
            z_exgk1 = z_exgk2 = z_exgk0 = ZQ_complex
            Zb0, Zb1, Zb2 = (Zbr(ppci_0, baseZ0, bus_fault_ppc, bus_ext_ppc),
                             Zbr(ppci_1, baseZ1, bus_fault_ppc, bus_ext_ppc),
                             Zbr(ppci_2, baseZ2, bus_fault_ppc, bus_ext_ppc))
            same_bus = (bus_ext_ppc == bus_fault_ppc)

    else:
        # --- No transformer (direct connection to external grid)
        z_exgk1 = z_exgk2 = z_exgk0 = ZQ_complex
        Zb0, Zb1, Zb2 = (Zbr(ppci_0, baseZ0, bus_fault_ppc, bus_ext_ppc),
                         Zbr(ppci_1, baseZ1, bus_fault_ppc, bus_ext_ppc),
                         Zbr(ppci_2, baseZ2, bus_fault_ppc, bus_ext_ppc))
        same_bus = (bus_ext_ppc == bus_fault_ppc)

    # --- Combine source and network impedances
    Z1 = np.where(same_bus, z_exgk1, Zb1 + z_exgk1)
    Z2 = np.where(same_bus, z_exgk2, Zb2 + z_exgk2)
    Z0 = np.where(same_bus, z_exgk0, Zb0 + z_exgk0)

    # --- Calculate symmetrical component currents depending on fault type
    if fault_type == "LLL":  # Three-phase fault
        I1, I2, I0 = Eg / Z1, np.zeros_like(I1), np.zeros_like(I1)
    elif fault_type == "LG":  # Single line-to-ground fault
        I0 = I1 = I2 = Eg / (Z1 + Z2 + Z0)
    elif fault_type == "LL":  # Line-to-line fault
        I1 = Eg / (Z1 + Z2)
        I2 = -I1
        I0 = np.zeros_like(I1)
    elif fault_type == "LLG":  # Double line-to-ground fault
        I1 = Eg / (Z1 + (Z2 * Z0) / (Z2 + Z0))
        I2 = -I1 * Z0 / (Z2 + Z0)
        I0 = -I1 * Z2 / (Z2 + Z0)

    # --- Convert sequence currents to phase quantities and store results
    results = []
    for i, ext_bus in enumerate(ext_grids.bus.values):
        I_phase = sequence_to_phase(np.array([I0[i], I1[i], I2[i]], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase
        V_phase_kv = vn_net[i] / np.sqrt(3)

        # Calculate apparent power per phase and total
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva

        results.append({
            "bus": ext_bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss": skss,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
        })

    # --- Store final results in the network
    net.res_ext_grid_sc = pd.DataFrame(results)


In [ ]:
#VECTORIZED 3.0 - WORKS FOR EVERY CASE AND FAULT
def _get_ext_grid_results(net, ppci_0, ppci_1, ppci_2, bus):

    # --- Map the PandaPower bus index to the corresponding PPC bus index
    bus_lookup = net._pd2ppc_lookups["bus"]
    bus_fault_ppc = bus_lookup[bus]

    # --- Select all in-service external grids
    ext_grids = net.ext_grid[net.ext_grid.get("in_service", True)].copy()
    if ext_grids.empty:
        # No external grids found – return NaN placeholders for results
        net.res_ext_grid_sc = pd.DataFrame([{
            "bus": np.nan,
            "ikss_a_ka": np.nan, "ikss_b_ka": np.nan, "ikss_c_ka": np.nan,
            "ikss_a_degree": np.nan, "ikss_b_degree": np.nan, "ikss_c_degree": np.nan,
            "skss": np.nan, "skss_a_mva": np.nan, "skss_b_mva": np.nan, "skss_c_mva": np.nan,
            "p_a_mw": np.nan, "q_a_mvar": np.nan,
            "p_b_mw": np.nan, "q_b_mvar": np.nan,
            "p_c_mw": np.nan, "q_c_mvar": np.nan
        }])
        return

    # --- Gather external grid parameters
    bus_ext_ppc = bus_lookup[ext_grids.bus.values]
    vn_net = net.bus.loc[ext_grids.bus.values, "vn_kv"].values
    s_sc = ext_grids.s_sc_max_mva.values
    rx = ext_grids.rx_max.values
    fault_type = net["_options"]["fault"]
    cmax = ppci_1["bus"][bus_ext_ppc, C_MAX]

    # --- Calculate equivalent source voltage (Eg) and impedance (ZQ)
    Eg = cmax * vn_net / np.sqrt(3)
    ZQ = (vn_net ** 2) / s_sc
    RQ = ZQ * rx / np.sqrt(1 + rx ** 2)
    XQ = ZQ / np.sqrt(1 + rx ** 2)
    ZQ_complex = cmax * (RQ + 1j * XQ)

    # --- Calculate base impedances for each sequence network
    baseZ0 = (ppci_0["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_0["baseMVA"]
    baseZ1 = (ppci_1["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_1["baseMVA"]
    baseZ2 = (ppci_2["bus"][bus_fault_ppc, BASE_KV] ** 2) / ppci_2["baseMVA"]

    # --- Helper: Compute equivalent impedance between fault bus and external grid bus
    def Zbr(ppci, baseZ, bus_fault_ppc, bus_ext_ppc):
        Zbus = ppci["internal"]["Zbus"]
        Z_from_from = Zbus[bus_fault_ppc, bus_fault_ppc]

        # Handle single or multiple external grid buses
        if np.isscalar(bus_ext_ppc):
            Z_to_to = Zbus[bus_ext_ppc, bus_ext_ppc]
            Z_from_to = Zbus[bus_fault_ppc, bus_ext_ppc]
        else:
            Z_to_to = np.diag(Zbus[bus_ext_ppc][:, bus_ext_ppc])
            Z_from_to = Zbus[bus_fault_ppc, bus_ext_ppc]

        Zeq = Z_from_from + Z_to_to - 2 * Z_from_to
        Zeq = np.atleast_1d(Zeq)
        Zeq[np.isclose(Zeq, 0)] = 1e-12  # Avoid numerical singularities
        return Zeq * baseZ

    # --- Check for transformers connected to the external grid
    trafo = net.trafo[
        net.trafo["hv_bus"].isin(ext_grids.bus.values) |
        net.trafo["lv_bus"].isin(ext_grids.bus.values)
    ]

    if not trafo.empty:
        # --- Extract transformer data (assuming one transformer per ext. grid)
        tr = trafo.iloc[0]
        vn_hv, vn_lv = float(tr.vn_hv_kv), float(tr.vn_lv_kv)
        sn = float(tr.sn_mva)

        # Transformer nominal ratio
        t_nominal = vn_hv / vn_lv
        hv_bus, lv_bus = bus_lookup[tr.hv_bus], bus_lookup[tr.lv_bus]
        branch = ppci_1["branch"]
        # Find transformer branch (for tap correction)
        mask = ((branch[:, F_BUS] == hv_bus) & (branch[:, T_BUS] == lv_bus)) | \
               ((branch[:, F_BUS] == lv_bus) & (branch[:, T_BUS] == hv_bus))


        # --- Transformer impedance (referenced to LV side)
        ZT1 = (ppci_1["branch"][mask, BR_R] + 1j * ppci_1["branch"][mask, BR_X]) * baseZ1
        ZT2 = (ppci_2["branch"][mask, BR_R] + 1j * ppci_2["branch"][mask, BR_X]) * baseZ2
        #THIS APPROACH IS NOT WORKING FOR ZERO SEQUENCE IMPEDANCE - ONLY BUS APPROACH - EXPLAINED DOWN
        ZTLVK_1, ZTLVK_2= ZT1, ZT2

        ZQt = ZQ_complex / (t_nominal ** 2)  # Refer network impedance to LV side

        # --- Total equivalent impedance (network + transformer)
        z_exgk1 = ZQt + ZTLVK_1
        #OR IT CAN BE DIRECTLY CALCULATED: z_exgk1 = (ppci_1["bus"][lv_bus, R_EQUIV] + 1j * ppci_2["bus"][lv_bus, X_EQUIV]) * baseZ1
        z_exgk2 = ZQt + ZTLVK_2
        # OR IT CAN BE DIRECTLY CALCULATED: z_exgk2 = (ppci_2["bus"][lv_bus, R_EQUIV] + 1j * ppci_2["bus"][lv_bus, X_EQUIV]) * baseZ2
        z_exgk0 = (ppci_0["bus"][bus_fault_ppc, R_EQUIV] + 1j * ppci_0["bus"][bus_fault_ppc, X_EQUIV]) * baseZ0
        # ONLY THIS APPROACH CAN BE USED FOR ZERO SEQUENCE IMPEDANCE IN ORDER TO BE CALCULATED CORRECTLY

        # Determine if fault is on LV side of transformer
        if lv_bus <= bus_fault_ppc:
            Eg = Eg / t_nominal  # Adjust voltage to LV side
            vn_net = net.bus.loc[ext_grids.bus.values, "vn_kv"].values / t_nominal
            Zb1, Zb2 = ( Zbr(ppci_1, baseZ1, bus_fault_ppc, lv_bus), Zbr(ppci_2, baseZ2, bus_fault_ppc, lv_bus))
            same_bus = (lv_bus == bus_fault_ppc)
            # we need to compare low votlafe bus of tgransdfomer as reference place for branch impedance that we are serarcig if fault is  not happenign at lv bus
            Z1 = np.where(same_bus, z_exgk1, Zb1 + z_exgk1)
            Z2 = np.where(same_bus, z_exgk2, Zb2 + z_exgk2)
            Z0 =  z_exgk0

        else:
            # Fault before transformer- if fault is  happenig before transformer (on high voltage bus, then transformer impedance should not be considreded)
            #only the grid imepdance matters =>the fault is happening at grid conenction bus
            z_exgk1 = z_exgk2 = z_exgk0 = ZQ_complex

            Z1 = z_exgk1
            Z2 = z_exgk2
            Z0 = z_exgk0

    else:
        # --- No transformer (direct connection of net  to external grid)- if fault is far away from external grid's bus or just at external grid's bus
        z_exgk1 = z_exgk2 = z_exgk0 =  ZQ_complex
        Zb0, Zb1, Zb2 = (Zbr(ppci_0, baseZ0, bus_fault_ppc, bus_ext_ppc),
                         Zbr(ppci_1, baseZ1, bus_fault_ppc, bus_ext_ppc),
                         Zbr(ppci_2, baseZ2, bus_fault_ppc, bus_ext_ppc))
        same_bus = (bus_ext_ppc == bus_fault_ppc)
        # --- Combine source and network impedances
        Z1 = np.where(same_bus, z_exgk1, Zb1 + z_exgk1)
        Z2 = np.where(same_bus, z_exgk2, Zb2 + z_exgk2)
        Z0 = np.where(same_bus, z_exgk2, Zb0 + z_exgk0)


    # --- Calculate symmetrical component currents depending on fault type
    if fault_type == "LLL":  # Three-phase fault
        I1 = Eg / Z1
        I2 = np.zeros_like(I1)
        I0 = np.zeros_like(I1)
    elif fault_type == "LG":  # Single line-to-ground fault
        I0 = I1 = I2 = Eg / (Z1 + Z2 + Z0)
    elif fault_type == "LL":  # Line-to-line fault
        I1 = Eg / (Z1 + Z2)
        I2 = -I1
        I0 = np.zeros_like(I1)
    elif fault_type == "LLG":  # Double line-to-ground fault
        I1 = Eg / (Z1 + (Z2 * Z0) / (Z2 + Z0))
        I2 = -I1 * Z0 / (Z2 + Z0)
        I0 = -I1 * Z2 / (Z2 + Z0)

    # --- Convert sequence currents to phase quantities and store results
    results = []
    for i, ext_bus in enumerate(ext_grids.bus.values):
        I_phase = sequence_to_phase(np.array([I0[i], I1[i], I2[i]], dtype=complex)).squeeze()
        I_a, I_b, I_c = I_phase
        V_phase_kv = vn_net[i] / np.sqrt(3)

        # Calculate apparent power per phase and total
        skss_a_mva = V_phase_kv * abs(I_a)
        skss_b_mva = V_phase_kv * abs(I_b)
        skss_c_mva = V_phase_kv * abs(I_c)
        skss = skss_a_mva + skss_b_mva + skss_c_mva

        results.append({
            "bus": ext_bus,
            "ikss_a_ka": abs(I_a), "ikss_b_ka": abs(I_b), "ikss_c_ka": abs(I_c),
            "ikss_a_degree": np.angle(I_a, deg=True),
            "ikss_b_degree": np.angle(I_b, deg=True),
            "ikss_c_degree": np.angle(I_c, deg=True),
            "skss": skss,
            "skss_a_mva": skss_a_mva, "skss_b_mva": skss_b_mva, "skss_c_mva": skss_c_mva,
            "p_a_mw": skss_a_mva * np.cos(np.angle(I_a)),
            "q_a_mvar": skss_a_mva * np.sin(np.angle(I_a)),
            "p_b_mw": skss_b_mva * np.cos(np.angle(I_b)),
            "q_b_mvar": skss_b_mva * np.sin(np.angle(I_b)),
            "p_c_mw": skss_c_mva * np.cos(np.angle(I_c)),
            "q_c_mvar": skss_c_mva * np.sin(np.angle(I_c)),
        })

    # --- Store final results in the network
    net.res_ext_grid_sc = pd.DataFrame(results)